In [4]:
from tqdm import tqdm
import numpy as np
from scipy.stats import truncnorm

class GreedyRoutingAnnealer:
    def __init__(self, adjacency_matrix, coord=None):
        self.adjacency_matrix = adjacency_matrix
        self.adjacency_list = np.array([np.where(neigh)[0] for neigh in adjacency_matrix], dtype=object)
        self.N = adjacency_matrix.shape[0]
        self.K = adjacency_matrix.sum()
        self.num_grs_iter = int(np.ceil(np.log2(N-1)))
        
        self.coord = self.get_random_init(self.adjacency_matrix) if coord is None else coord
        self.distance = self.get_distance(self.coord)
        self.gcn = self.greedy_closest_neighbour(self.adjacency_list, self.distance)
        self.gd = self.greedy_destination(self.gcn)
        self.grs = self.greedy_routing_score(self.gd)

    def get_random_init(self, adjacency_matrix):
        N = adjacency_matrix.shape[0]
        clip_min, clip_max = 1, 2*np.log(self.N)
        loc, scale = 2.5*np.log(self.N), np.log(self.N) /2
        a, b = (clip_min - loc) / scale, (clip_max - loc) / scale
        rng = truncnorm(a=a, b=b, loc=loc, scale=scale)
        new_r = rng.rvs(self.N)
        new_r = np.sort(new_r)[::-1]
        new_theta = 2 * np.pi * np.random.rand(self.N)
        degree_order = np.argsort(np.sum(self.adjacency_matrix, 0)+np.random.rand(self.N))
        coord = np.empty((self.N, 2))
        coord[degree_order, 0] = new_r
        coord[:, 1] = new_theta
        return coord

    def get_distance(self, coord: np.ndarray) -> np.ndarray:
        d_theta = np.pi - np.abs(np.pi - np.abs(coord[:, 1]-coord[:, np.newaxis, 1]))
        distance = np.cosh(coord[:, 0]) * np.cosh(coord[:, np.newaxis, 0])
        distance -= np.sinh(coord[:, 0]) * np.sinh(coord[:, np.newaxis, 0]) * np.cos(d_theta)
        return distance

    def greedy_closest_neighbour(self, distance: np.ndarray) -> np.ndarray:
        gcn = np.array([neighbours[np.argmin(distance[neighbours], axis=0)] for neighbours in self.adjacency_list])
        np.fill_diagonal(gcn, np.arange(N))
        return gcn

    def greedy_destination(self, gcn: np.ndarray) -> np.ndarray:
        arange = np.arange(gcn.shape[1])
        for _ in range(self.num_grs_iter):
            gcn = gcn[gcn, arange]
        return gcn
    
    def greedy_routing_score(self, gd: np.ndarray) -> float:
        successful = np.sum(gd == np.arange(N))
        return (successful - N) / N / (N-1)
    
    def select_for_move(self, gd: np.ndarray) -> np.ndarray:
        return np.random.randint(self.N)
    
    def random_move(self, moved_vertex: int) -> np.ndarray:
        coord = self.coord.copy()
        clip_min, clip_max = 1, 2*np.log(self.N)
        loc, scale = self.coord[moved_vertex, 0], 1
        a, b = (clip_min - loc) / scale, (clip_max - loc) / scale
        coord[moved_vertex, 0] = truncnorm.rvs(a=a, b=b, loc=loc, scale=scale)
        coord[moved_vertex, 1] = np.random.normal(loc=self.coord[moved_vertex, 1], scale=np.pi/4) % (2*np.pi)
        return coord

    def recalc_distance(self, coord: np.ndarray, moved_vertex: int) -> np.ndarray:
        distance = self.distance.copy()
        d_theta = np.pi - np.abs(np.pi - np.abs(coord[moved_vertex, 1]-coord[:, 1]))
        dist = np.cosh(coord[moved_vertex, 0]) * np.cosh(coord[:, 0])
        dist -= np.sinh(coord[moved_vertex, 0]) * np.sinh(coord[:, 0]) * np.cos(d_theta)
        dist[moved_vertex] = 1
        distance[moved_vertex] = distance[:, moved_vertex] = dist
        return distance

    def recalc_greedy_routing(self, distance, moved_vertex):
        gcn = self.gcn.copy()
        gd = self.gd.copy()
        
        affected = np.append(self.adjacency_list[moved_vertex], moved_vertex)
        affected_next_hop = np.array([neighbours[np.argmin(distance[neighbours], axis=0)] for neighbours in self.adjacency_list[affected]])
        affected_next_hop[np.arange(affected.shape[0]), affected] = affected

        gcn[:, moved_vertex] = np.array([neighbours[np.argmin(distance[neighbours, moved_vertex], axis=0)] for neighbours in self.adjacency_list])
        gcn[affected] = affected_next_hop
        
        changed = np.any(affected_next_hop != self.gcn[affected], axis=0)
        changed[moved_vertex] = True
        
        gd[:, changed] = self.greedy_destination(gcn[:, changed])
        return gcn, gd

    def embed(self, steps=2**8, verbose=False):
        
        steps = int(steps)
        grs_array = np.empty(steps+1)
        log[0] = self.grs

        temperature = 1 / np.arange(1, steps+1)
        if verbose:
            temperature = tqdm(temperature)

        for step, temp in enumerate(temperature, start=1):
            moved_vertex = self.select_for_move(self.gd)
            coord = self.random_move(moved_vertex)
            distance = self.recalc_distance(coord, moved_vertex)
            gcn, gd = self.recalc_greedy_routing(distance, moved_vertex)
            grs = self.greedy_routing_score(gd)
            
            dE = (grs - self.grs) / N
            
            if 0 < dE or np.random.rand() < np.exp(dE / temp):
                self.coord = coord
                self.distance = distance
                self.gcn = gcn
                self.gd = gd
                self.grs = grs
            
            grs_array[step] = grs

        return grs_array